In [ ]:
!pip install langchain

### First LangChain Code
- Install langchain-ollama

In [ ]:
!pip install langchain-ollama

### First chat with Ollama using Langchain

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

template = PromptTemplate.from_template("What is the capital of {countryName}?")
prompt = template.invoke({"countryName": "France"})


chatTemplate = ChatPromptTemplate.from_messages()

print(prompt)

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )

res = llm_model.invoke(prompt)
print(res)



text='What is the capital of France?'
content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-03-31T14:38:04.67813882Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2231798628, 'load_duration': None, 'prompt_eval_count': 15, 'prompt_eval_duration': None, 'eval_count': 8, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019d4454-7c61-7813-a090-cb022bf0f38e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23}


### Different Types of Messages

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# prompt = "What is the capital of France?"  
# print(type(prompt))  # string

# humanMsg = HumanMessage(content="what is the capital of France?")
# print(type(humanMsg))

# print(humanMsg)

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )

# res = llm_model.invoke(humanMsg.content)
# print(res)
# print(type(res))




### ChatPromptTemplate

chatTemplate = ChatPromptTemplate.from_messages([
    ("human", "What is the capital of {countryName}?"),
    ("ai", "The capital of {countryName} is {cityName}"),
    ("human","What is the most popular city of {cityName}")

    # HumanMessage(content="What is the capital of {countryName}?"),
    # AIMessage(content= "The capital of {countryName} is {cityName}"),
    # HumanMessage(content= "What is the most popular city of {cityName}")
])

#res = llm_model.invoke(chatTemplate.invoke({"countryName": "India", "cityName": "Delhi"}))

chain = chatTemplate | llm_model

res = chain.invoke({"countryName": "India", "cityName": "Delhi"})

print(res)


content='Delhi itself is a city—but it\'s actually divided into two main parts:\n\n- **New Delhi**: The planned, northern part of Delhi, which serves as the **national capital territory** and houses government institutions (like the Parliament, Rashtrapati Bhavan, and Supreme Court). It\'s often what people mean when they refer to "Delhi" in an administrative or political context.\n\n- **Old Delhi**: The historic, southern part of Delhi, known for its Mughal-era landmarks like the Red Fort, India Gate, Jama Masjid, and bustling markets like Chandni Chowk.\n\nSo, there isn’t a “most popular city *of* Delhi”—rather, **New Delhi and Old Delhi** are the two iconic regions *within* the larger city of **Delhi** (officially the **National Capital Territory of Delhi**, or NCT).\n\nDelhi is also India’s **second-most populous city** (after Mumbai), and it’s a major cultural, political, and economic hub.\n\nLet me know if you\'d like highlights of Old vs. New Delhi! 😊' additional_kwargs={} respo

### Understanding Tooling in LangChain

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompt_values import PromptValue
from pydantic import BaseModel

class AIResponse(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str    


@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

@tool
def calculate(expression: str) -> str:
    """Calculate the result of a mathematical expression
    
    Args:
        expression: A string representing a mathematical expression
    
    Returns:
        A string representing the result of the calculation
    """
    return str(eval(expression))



def podTestAgent(prompt: ChatPromptTemplate, format: any, countryName: str ):  

    promptValue = prompt.invoke({"format": format, "countryName": countryName})  
    messages = list(promptValue.to_messages())

    tools = [get_weather_city, calculate]
    tools_by_name = {}   # str: StructureTool  {"get_weather_city":  StructuredTool}

    for tool in tools:
        tools_by_name[f"{tool.name}"] = tool   

    llm_model = ChatOllama(
        base_url="https://ollama.com",
        model="qwen3-coder-next:cloud",
        client_kwargs=  {  
            "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
            }
        )

    llm_model_with_tools = llm_model.bind_tools(tools)


    while True:
        res = llm_model_with_tools.invoke(messages)
        print(res)
        #print(str(prompt.invoke()))

        if not res.tool_calls:
            return res.content
        
        messages.append(res)

        # extract tool_call info
        tools_call = res.tool_calls  # list of tools
        for tool in tools_call:
            tool_name = tool["name"]
            tool_args =tool["args"]
            tool_id = tool["id"]
            tool_result = tools_by_name[tool_name].invoke(tool_args)
            #print(tool_result)

            toolmsg = ToolMessage(content=str(tool_result), tool_call_id=tool_id)
            messages.append(toolmsg)          
   


prompt = ChatPromptTemplate.from_messages([   
     ("system", "Provide response in JSON format {format}"),
    ("human", "What is the weather in {countryName} right now?   what is it's capital? ")    
    ])

res = podTestAgent( prompt,  AIResponse.model_json_schema(), "India")
print(res)


#print(type(res))


In [28]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompt_values import PromptValue
from pydantic import BaseModel
from langchain_core.runnables import RunnableSequence

class AIResponse(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str 

prompt = ChatPromptTemplate.from_messages([   
     ("system", "Provide response in JSON format {format}"),
    ("human", "What is the weather in {countryName} right now?   what is it's capital? ")    
    ])

#input  {"format": AIResponse.model_json_schema(), "countryName": "India"}
#Output of ChatPromptTemplate: promptValue
#promptValue = prompt.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"}) 

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )  #.with_structured_output(AIResponse)

#chaining of Runnables:
#chain =  prompt | llm_model
chain = RunnableSequence(prompt, llm_model)
res = chain.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"})

#RunnableSequnce
#RunnableSequence(prompt, llm_model)

#res = llm_model.invoke(promptValue)



print(res)





content='{\n  "countryName": "India",\n  "countryWeather": "I cannot provide real-time weather data. Please check a reliable weather service like Weather.com, AccuWeather, or a weather app for current conditions in India.",\n  "countryCapital": "New Delhi"\n}' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-02T14:41:36.428492641Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1034604247, 'load_duration': None, 'prompt_eval_count': 118, 'prompt_eval_duration': None, 'eval_count': 58, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019d4ea4-786f-7392-8dcc-77fc735070a9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 118, 'output_tokens': 58, 'total_tokens': 176}


### Understand Parallel Execution of runnables

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompt_values import PromptValue
from pydantic import BaseModel
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda

class AIResponse(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str 

prompt = ChatPromptTemplate.from_messages([   
     ("system", "Provide response in JSON format {format}"),
    ("human", "What is the weather in {countryName} right now?   what is it's capital? ")    
    ])

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )  #.with_structured_output(AIResponse)

prompt2 = PromptTemplate.from_template("who is the Prime minister of Canada?")

#chaining of Runnables:
#chain =  prompt | llm_model
chain1 = RunnableSequence(prompt, llm_model)
chain2= RunnableSequence(prompt2, llm_model)

finalchain = RunnableParallel(
    country_info=chain1,   # result of chain1 stored under key "country_info"
    pm_info=chain2  )

res = finalchain.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"})
#res = chain.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"})

#RunnableSequnce
#RunnableSequence(prompt, llm_model)

#res = llm_model.invoke(promptValue)

def add(a: int, b: int) -> int:
    return a+b



#res =  | llm_model


print(res)





{'country_info': AIMessage(content='{"countryName": "India", "countryWeather": "Partly cloudy", "countryCapital": "New Delhi"}', additional_kwargs={}, response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-02T14:53:01.343123729Z', 'done': True, 'done_reason': 'stop', 'total_duration': 702128727, 'load_duration': None, 'prompt_eval_count': 118, 'prompt_eval_duration': None, 'eval_count': 25, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'}, id='lc_run--019d4eae-ecfe-7d50-9603-643e61fcdf49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 25, 'total_tokens': 143}), 'pm_info': AIMessage(content='As of now, the Prime Minister of Canada is **Justin Trudeau**. He has held the position since November 4, 2015, and was most recently re-elected in the 2021 federal election. He leads the Liberal Party of Canada.\n\nNote: Political leadership can change, so for the most u

In [ ]:
from langchain_core.runnables import RunnableLambda

#approach1
# def add(a: int, b: int) -> int:
#     return a+b

# r1 = RunnableLambda(lambda x: add(**x))
# res = r1.invoke({"a": 4, "b": 6})


#approach2
def add(input: dict) -> int:
    return input["a"]+input["b"]

r1 = RunnableLambda(add)
res = r1.invoke({"a": 4, "b": 6})

print(res)

#c = r1 | llm_model

10


### Easiest way to create an Agents

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage
from pydantic import BaseModel

#model
llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    ) 


#respone format
class ResponseOutput(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str

#tool
@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

@tool
def calculate(expression: str) -> str:
    """Calculate the result of a mathematical expression
    
    Args:
        expression: A string representing a mathematical expression
    
    Returns:
        A string representing the result of the calculation
    """
    return str(eval(expression))

tool_list = [calculate,get_weather_city]

agent = create_agent(
    name= "PodTest Agent"
    model=llm_model,
    tools=tool_list,
    #system_prompt="Act as an SDET & provide answer in JSOn format"
    #system_prompt=SystemMessage("Act as an SDET & provide answer in JSOn format")
    response_format=ToolStrategy(ResponseOutput)
)


prompt = PromptTemplate.from_template("What is the weather in India? What is the capital of India?")
promptValue = prompt.invoke({})

res = agent.invoke({"messages": str(promptValue)})
print(res["structured_response"])



countryName='India' countryWeather='The weather in India is sunny.' countryCapital='New Delhi'


### BDD testcase AI Agent

In [2]:
# ============================================================
# BDD Test Case Generator Agent — Latest LangChain
# pip install -qU langchain "langchain[anthropic]"
# ============================================================

import os
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# ──────────────────────────────────────────────
# TOOLS
# ──────────────────────────────────────────────

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5d878c6be2d14295ab07c8b714799ccc.li1Ulo3fcWSP1YI3Rq8w3Ru2'}
        }
    ) 

@tool
def parse_requirements(requirements: str) -> str:
    """
    Parses raw requirement text and extracts:
    - Actors / users involved
    - Actions / behaviors expected
    - Expected outcomes / acceptance criteria
    Returns a structured summary used for BDD generation.
    """
    lines = [l.strip() for l in requirements.strip().splitlines() if l.strip()]
    summary = "\n".join(f"  - {line}" for line in lines)
    return (
        f"Parsed Requirements:\n{summary}\n\n"
        f"Total requirement lines identified: {len(lines)}\n"
        f"Ready for BDD test case generation."
    )


@tool
def generate_bdd_scenarios(parsed_requirements: str) -> str:
    """
    Takes parsed requirements and generates comprehensive BDD test cases
    in Gherkin format (Feature / Scenario / Given / When / Then).
    Covers happy paths, edge cases, and negative scenarios.
    The agent will produce the actual Gherkin content.
    """
    return (
        f"Generating BDD scenarios for:\n{parsed_requirements}\n\n"
        "Instruction to agent: Now write complete Gherkin BDD test cases "
        "covering all happy paths, edge cases, and error/negative scenarios."
    )


@tool
def validate_gherkin_syntax(gherkin_text: str) -> str:
    """
    Validates that the BDD test cases follow proper Gherkin syntax rules.
    Checks for required keywords: Feature, Scenario, Given, When, Then.
    Returns 'VALID' or a list of syntax issues found.
    """
    required_keywords = ["Feature:", "Scenario:", "Given", "When", "Then"]
    issues = []

    for kw in required_keywords:
        if kw not in gherkin_text:
            issues.append(f"Missing required Gherkin keyword: '{kw}'")

    lines = gherkin_text.strip().splitlines()
    valid_starts = (
        "Feature:", "Background:", "Scenario:", "Scenario Outline:",
        "Given", "When", "Then", "And", "But", "Examples:", "|", "#", "@", ""
    )
    for i, line in enumerate(lines, 1):
        stripped = line.strip()
        if stripped and not any(stripped.startswith(v) for v in valid_starts):
            issues.append(f"Line {i}: Unrecognized Gherkin syntax → '{stripped[:60]}'")

    if issues:
        return "VALIDATION ISSUES FOUND:\n" + "\n".join(f"  ⚠ {i}" for i in issues)
    return "✅ VALID — All Gherkin syntax checks passed."


@tool
def export_feature_file(gherkin_text: str, feature_filename: str = "generated_tests.feature") -> str:
    """
    Exports the final validated BDD test cases as a .feature file.
    Saves the file to disk and returns the file path and content summary.
    """
    output_path = os.path.join(os.getcwd(), feature_filename)
    header = (
        "# ================================================\n"
        "# Auto-generated BDD Feature File\n"
        "# Generated by: BDD Agent (LangChain + Claude)\n"
        "# ================================================\n\n"
    )
    content = header + gherkin_text.strip()

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)

    scenario_count = gherkin_text.count("Scenario:")
    return (
        f"✅ Feature file exported successfully!\n"
        f"   Path     : {output_path}\n"
        f"   Scenarios: {scenario_count} scenario(s) written\n"
        f"   File     : {feature_filename}"
    )


# ──────────────────────────────────────────────
# SYSTEM PROMPT
# ──────────────────────────────────────────────

BDD_SYSTEM_PROMPT = """
You are a senior QA engineer and BDD expert specialising in writing
Behaviour-Driven Development test cases using Gherkin syntax.

## Your Workflow (follow this ORDER every time):

1. **Parse** — Call `parse_requirements` with the user's raw requirements.
2. **Generate** — Call `generate_bdd_scenarios` with the parsed output,
   then write thorough Gherkin BDD test cases that include:
     - A `Feature:` block with a meaningful description
     - A `Background:` section (if common preconditions exist)
     - **Happy path** scenarios (standard successful flows)
     - **Edge case** scenarios (boundary values, unusual inputs)
     - **Negative / error** scenarios (invalid inputs, failures)
     - `Scenario Outline:` with `Examples:` tables where data-driven
       testing makes sense
3. **Validate** — Call `validate_gherkin_syntax` on your generated Gherkin.
   Fix any issues before proceeding.
4. **Export** — Call `export_feature_file` to save the final output.
5. **Summarise** — Tell the user what was generated: feature name,
   number of scenarios, and any important design decisions.

## Gherkin Rules:
- Start every file with `Feature:` followed by a 1–2 line description
- Use `Scenario:` for individual test cases
- Steps must begin with `Given`, `When`, `Then`, `And`, or `But`
- `Given` = precondition/setup
- `When`  = the action/event being tested
- `Then`  = the expected outcome/assertion
- Use data tables (`|`) and `Scenario Outline` for parameterised tests
- Add `@tags` above scenarios to categorise them (e.g. @smoke, @regression)
- Keep scenarios independent — no shared state between scenarios

Be thorough, precise, and cover all realistic test scenarios.
"""

# ──────────────────────────────────────────────
# CREATE AGENT  (new LangChain API)
# ──────────────────────────────────────────────

agent = create_agent(
    #model="claude-sonnet-4-6",          # or "anthropic:claude-opus-4-6"
    model=llm_model,
    tools=[
        parse_requirements,
        generate_bdd_scenarios,
        validate_gherkin_syntax,
        export_feature_file,
    ],
    system_prompt=BDD_SYSTEM_PROMPT,
)


# ──────────────────────────────────────────────
# RUN THE AGENT
# ──────────────────────────────────────────────

def generate_bdd_tests(requirements: str) -> str:
    """Invoke the BDD agent with a requirements string."""
    result = agent.invoke(
        
        
        {
        "messages": [
            {
                "role": "user",
                "content": f"Please generate BDD test cases for the following requirements:\n\n{requirements}"
            }           

        ]
        },
        "user_prefernces": {}
    
    
    )
    # Extract the final assistant message
    messages = result.get("messages", [])
    for msg in reversed(messages):
        if hasattr(msg, "content") and msg.content:
            return msg.content
    return str(result)


# ──────────────────────────────────────────────
# EXAMPLE USAGE
# ──────────────────────────────────────────────

if __name__ == "__main__":
    sample_requirements = """
    User Login Feature:
    - Users must be able to log in with a valid email and password.
    - After 3 consecutive failed login attempts, the account should be locked for 15 minutes.
    - Users can reset their password via a link sent to their registered email.
    - Passwords must be at least 8 characters, contain 1 uppercase letter and 1 number.
    - Logged-in sessions should expire after 30 minutes of inactivity.
    - Users should see a descriptive error message for invalid credentials.
    """

    print("🤖 BDD Agent starting...\n")
    output = generate_bdd_tests(sample_requirements)
    print(output)

🤖 BDD Agent starting...

### ✅ Summary

- **Feature Name**: User Login and Authentication
- **Number of Scenarios**: 11 scenarios + 2 `Scenario Outline` templates (yielding 18+ total test cases)
- **Key Design Decisions**:
  - Used a `Background` section to set up shared test users and system policy context.
  - Separated `@smoke`, `@positive`, `@negative`, `@edge`, and `@regression` tags for test suite filtering.
  - Included `Scenario Outline`s for data-driven password complexity and login error validation.
  - Ensured privacy-sensitive behavior: no confirmation of email existence during password reset.
  - Verified both happy paths and failure modes (lockout, invalid passwords, session expiry).
  - All syntax validated — ready for execution with Cucumber or similar frameworks.

The `user_login.feature` file has been saved to:  
`e:\Akhil\PodTest\Clients\Self\Bootcamp\FullStackSDET\FSLangChain\user_login.feature`

Let me know if you'd like this extended to include API-level testing, 

### usage showing advanced concepts

In [ ]:
# ============================================================
# BDD Test Case Generator Agent — Latest LangChain (v1.x)
# Official docs: https://docs.langchain.com/oss/python/langchain/overview
#
# pip install -qU langchain "langchain[anthropic]" langgraph
# ============================================================


import os
from dataclasses import dataclass
from typing import Any, Callable
from typing_extensions import NotRequired

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import (
    AgentMiddleware,
    ModelRequest,
    ModelResponse,
    hook_config,
)
from langchain.messages import AIMessage, SystemMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langgraph.types import Command


llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    ) 
# ──────────────────────────────────────────────────────────────
# 1. STATE SCHEMA  (extends AgentState with BDD-specific fields)
# ──────────────────────────────────────────────────────────────

class BDDAgentState(AgentState):
    """Custom short-term state persisted across every agent step."""
    session_id:        NotRequired[str]   # identifies the test session
    raw_requirements:  NotRequired[str]   # original requirements text
    parsed_summary:    NotRequired[str]   # output of parse_requirements tool
    generated_gherkin: NotRequired[str]   # BDD Gherkin produced by agent
    validation_result: NotRequired[str]   # result of syntax validation
    feature_file_path: NotRequired[str]   # path of exported .feature file
    model_call_count:  NotRequired[int]   # tracks LLM calls (for guard rail)
    tool_call_log:     NotRequired[list]  # audit log of every tool call


# ──────────────────────────────────────────────────────────────
# 2. CONTEXT SCHEMA  (runtime context injected at invoke time)
# ──────────────────────────────────────────────────────────────

@dataclass
class BDDContext:
    """Per-invocation context: who is running this session and how."""
    user_id:          str              # e.g. "qa_engineer_42"
    project_name:     str              # e.g. "PaymentService"
    output_directory: str = "."        # where .feature files are saved
    max_model_calls:  int = 20         # safety cap on LLM calls


# ──────────────────────────────────────────────────────────────
# 3. TOOLS  (the agent's action repertoire)
# ──────────────────────────────────────────────────────────────

@tool
def parse_requirements(
    requirements: str,
    runtime: ToolRuntime,
) -> Command:
    """
    Parse raw requirements text.
    Extracts actors, actions, and expected outcomes.
    Saves the summary back into state for downstream tools.
    """
    lines = [l.strip() for l in requirements.strip().splitlines() if l.strip()]
    summary_lines = "\n".join(f"  [{i+1}] {line}" for i, line in enumerate(lines))
    summary = (
        f"Parsed {len(lines)} requirement(s):\n{summary_lines}\n\n"
        "Ready for BDD scenario generation."
    )
    return Command(update={
        "raw_requirements":  requirements,
        "parsed_summary":    summary,
        "messages": [
            ToolMessage(summary, tool_call_id=runtime.tool_call_id)
        ],
    })


@tool
def generate_bdd_scenarios(
    parsed_requirements: str,
    runtime: ToolRuntime,
) -> Command:
    """
    Trigger BDD scenario generation.
    Signals the agent to write Gherkin covering happy paths,
    edge cases, and negative/error scenarios.
    """
    instruction = (
        f"Requirements received:\n{parsed_requirements}\n\n"
        "Instruction: Write complete Gherkin BDD test cases now. "
        "Include Feature block, Background (if applicable), and Scenarios "
        "covering happy paths, edge cases, and negative scenarios. "
        "Use Scenario Outline + Examples for data-driven cases."
    )
    return Command(update={
        "messages": [
            ToolMessage(instruction, tool_call_id=runtime.tool_call_id)
        ],
    })


@tool
def save_gherkin_to_state(
    gherkin_text: str,
    runtime: ToolRuntime,
) -> Command:
    """
    Persist the generated Gherkin BDD text into agent state.
    Call this immediately after writing the BDD scenarios.
    """
    return Command(update={
        "generated_gherkin": gherkin_text,
        "messages": [
            ToolMessage(
                "✅ Gherkin saved to state.",
                tool_call_id=runtime.tool_call_id,
            )
        ],
    })


@tool
def validate_gherkin_syntax(
    gherkin_text: str,
    runtime: ToolRuntime,
) -> Command:
    """
    Validate that the BDD test cases follow proper Gherkin syntax.
    Checks for required keywords: Feature, Scenario, Given, When, Then.
    Returns VALID or a list of syntax issues.
    """
    required = ["Feature:", "Scenario:", "Given", "When", "Then"]
    issues = [f"Missing keyword: '{kw}'" for kw in required if kw not in gherkin_text]

    valid_starts = (
        "Feature:", "Background:", "Scenario:", "Scenario Outline:",
        "Given", "When", "Then", "And", "But", "Examples:", "|", "#", "@", "",
    )
    for i, line in enumerate(gherkin_text.splitlines(), 1):
        s = line.strip()
        if s and not any(s.startswith(v) for v in valid_starts):
            issues.append(f"Line {i} — unrecognised keyword: '{s[:60]}'")

    result = (
        "✅ VALID — All Gherkin syntax checks passed."
        if not issues
        else "⚠️ Issues found:\n" + "\n".join(f"  • {i}" for i in issues)
    )
    return Command(update={
        "validation_result": result,
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
def export_feature_file(
    gherkin_text: str,
    runtime: ToolRuntime[BDDContext, BDDAgentState],
) -> Command:
    """
    Export the validated BDD test cases as a .feature file.
    Uses context.output_directory and context.project_name from runtime.
    """
    project   = runtime.context.project_name
    out_dir   = runtime.context.output_directory
    user_id   = runtime.context.user_id
    filename  = f"{project.lower().replace(' ', '_')}_bdd_tests.feature"
    full_path = os.path.join(out_dir, filename)

    header = (
        "# =====================================================\n"
        f"# Project    : {project}\n"
        f"# Generated by: BDD Agent | User: {user_id}\n"
        "# =====================================================\n\n"
    )
    content = header + gherkin_text.strip()

    os.makedirs(out_dir, exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)

    scenario_count = gherkin_text.count("Scenario:")
    message = (
        f"✅ Feature file exported!\n"
        f"   Path     : {full_path}\n"
        f"   Scenarios: {scenario_count}\n"
    )
    return Command(update={
        "feature_file_path": full_path,
        "messages": [ToolMessage(message, tool_call_id=runtime.tool_call_id)],
    })


# ──────────────────────────────────────────────────────────────
# 4. MIDDLEWARE  (class-based, multiple hooks)
# ──────────────────────────────────────────────────────────────

class BDDAgentMiddleware(AgentMiddleware[BDDAgentState]):
    """
    Full-featured middleware for the BDD agent.

    Hooks implemented:
      before_agent  — initialise session metadata
      before_model  — enforce call-count limit + log state
      after_model   — increment call counter
      wrap_model_call — retry on transient errors
      wrap_tool_call  — audit-log every tool invocation
      after_agent   — print final session summary
    """

    # Tell LangChain this middleware owns BDDAgentState extensions
    state_schema = BDDAgentState

    # ── before_agent ──────────────────────────────────────────
    def before_agent(
        self,
        state: BDDAgentState,
        runtime: Runtime,
    ) -> dict[str, Any] | None:
        """Runs once per invocation, before any model call."""
        session = state.get("session_id", "bdd-session-001")
        print(f"\n{'='*60}")
        print(f"  🤖 BDD Agent started  |  Session: {session}")
        print(f"{'='*60}\n")
        return {
            "model_call_count": state.get("model_call_count", 0),
            "tool_call_log":    state.get("tool_call_log", []),
        }

    # ── before_model ──────────────────────────────────────────
    @hook_config(can_jump_to=["end"])
    def before_model(
        self,
        state: BDDAgentState,
        runtime: Runtime,
    ) -> dict[str, Any] | None:
        """Runs before each LLM call. Enforces call-count guard rail."""
        call_count = state.get("model_call_count", 0)

        # Read max_model_calls from runtime context if available
        max_calls = 20
        if runtime and hasattr(runtime, "context") and runtime.context:
            max_calls = getattr(runtime.context, "max_model_calls", 20)

        print(f"  📞 Model call #{call_count + 1}/{max_calls} "
              f"| Messages in state: {len(state['messages'])}")

        if call_count >= max_calls:
            print(f"  🚫 Call limit ({max_calls}) reached — stopping agent.")
            return {
                "messages": [AIMessage(
                    f"⚠️ Reached the maximum allowed model calls ({max_calls}). "
                    "Stopping here. Please review what has been generated so far."
                )],
                "jump_to": "end",
            }
        return None

    # ── after_model ───────────────────────────────────────────
    def after_model(
        self,
        state: BDDAgentState,
        runtime: Runtime,
    ) -> dict[str, Any] | None:
        """Runs after each LLM response. Increments call counter."""
        last = state["messages"][-1]
        tool_names = (
            [tc["name"] for tc in last.tool_calls]
            if hasattr(last, "tool_calls") and last.tool_calls
            else []
        )
        count = state.get("model_call_count", 0) + 1
        print(f"  ✅ Model responded  | Call count now: {count}"
              + (f" | Tools requested: {tool_names}" if tool_names else ""))
        return {"model_call_count": count}

    # ── wrap_model_call ───────────────────────────────────────
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Wraps each LLM call with retry logic (up to 3 attempts)."""
        max_retries = 3
        for attempt in range(1, max_retries + 1):
            try:
                return handler(request)
            except Exception as exc:
                if attempt == max_retries:
                    print(f"  ❌ Model call failed after {max_retries} attempts: {exc}")
                    raise
                print(f"  ⚠️  Model call attempt {attempt} failed ({exc}). Retrying…")

    # ── wrap_tool_call ────────────────────────────────────────
    def wrap_tool_call(
        self,
        request,
        handler,
    ):
        """Wraps every tool execution with audit logging."""
        tool_name = request.tool_call.get("name", "unknown")
        tool_args = request.tool_call.get("args", {})

        print(f"\n  🔧 Tool call  → {tool_name}")
        if tool_args:
            for k, v in tool_args.items():
                preview = str(v)[:80] + ("…" if len(str(v)) > 80 else "")
                print(f"       {k}: {preview}")

        try:
            result = handler(request)
            print(f"  ✔  {tool_name} completed successfully.")
            return result
        except Exception as exc:
            print(f"  ✘  {tool_name} raised: {exc}")
            raise

    # ── after_agent ───────────────────────────────────────────
    def after_agent(
        self,
        state: BDDAgentState,
        runtime: Runtime,
    ) -> dict[str, Any] | None:
        """Runs once when the agent finishes. Prints a summary."""
        print(f"\n{'='*60}")
        print("  📋 BDD Agent Session Summary")
        print(f"{'='*60}")
        print(f"  Total model calls : {state.get('model_call_count', 0)}")
        print(f"  Feature file      : {state.get('feature_file_path', 'not exported')}")
        print(f"  Validation        : {state.get('validation_result', 'not run')}")
        print(f"{'='*60}\n")
        return None


# ──────────────────────────────────────────────────────────────
# 5. SYSTEM PROMPT
# ──────────────────────────────────────────────────────────────

BDD_SYSTEM_PROMPT = """
You are a senior QA engineer and BDD expert specialising in Gherkin test cases.

## Workflow (always follow this order):
1. Call `parse_requirements` with the raw requirements.
2. Call `generate_bdd_scenarios` with the parsed summary.
3. Write comprehensive Gherkin BDD test cases covering:
   - Feature block with description
   - Background (if common preconditions exist)
   - Happy path scenarios
   - Edge case scenarios (boundary values, unusual inputs)
   - Negative / error scenarios (invalid input, auth failures, etc.)
   - Scenario Outline + Examples tables for data-driven tests
4. Call `save_gherkin_to_state` to persist the Gherkin you wrote.
5. Call `validate_gherkin_syntax` with your Gherkin text.
6. Fix any syntax issues reported, then call `export_feature_file`.
7. Summarise what was generated for the user.

## Gherkin Rules:
- Start with `Feature:` and a 1-2 line description
- `Given` = precondition, `When` = action, `Then` = expected outcome
- Each scenario must be fully independent
- Use `@tags` to categorise scenarios (e.g. @smoke, @regression, @negative)
- Use `Scenario Outline:` + `Examples:` for parameterised tests
"""


# ──────────────────────────────────────────────────────────────
# 6. IN-MEMORY CHECKPOINTER  (short-term thread-level memory)
# ──────────────────────────────────────────────────────────────

checkpointer = InMemorySaver()


# ──────────────────────────────────────────────────────────────
# 7. CREATE AGENT
# ──────────────────────────────────────────────────────────────

agent = create_agent(
    #model="claude-sonnet-4-6",            # swap to any supported model string
    model=llm_model,
    tools=[
        parse_requirements,
        generate_bdd_scenarios,
        save_gherkin_to_state,
        validate_gherkin_syntax,
        export_feature_file,
    ],
    system_prompt=BDD_SYSTEM_PROMPT,
    state_schema=BDDAgentState,           # custom state schema
    context_schema=BDDContext,            # runtime context schema
    middleware=[BDDAgentMiddleware()],     # class-based middleware
    checkpointer=checkpointer,            # InMemory checkpointer
    name="bdd_test_generator",
)


# ──────────────────────────────────────────────────────────────
# 8. RUNNER HELPER
# ──────────────────────────────────────────────────────────────

def generate_bdd_tests(
    requirements: str,
    project_name: str = "MyProject",
    user_id:      str = "qa_user",
    thread_id:    str = "thread-001",
    output_dir:   str = ".",
) -> str:
    """
    Invoke the BDD agent with requirements and context.

    Args:
        requirements: Plain-text requirements to convert to BDD.
        project_name: Used in the exported .feature file header.
        user_id:      Identifies the QA engineer for audit purposes.
        thread_id:    LangGraph thread ID — reuse to continue a conversation.
        output_dir:   Directory where the .feature file will be saved.

    Returns:
        The final assistant message as a string.
    """
    context = BDDContext(
        user_id=user_id,
        project_name=project_name,
        output_directory=output_dir,
        max_model_calls=20,
    )

    initial_state = {
        "messages": [{
            "role": "user",
            "content": (
                f"Generate complete BDD test cases for the following requirements:\n\n"
                f"{requirements}"
            ),
        }],
        "session_id": thread_id,
        "model_call_count": 0,
        "tool_call_log": [],
    }

    result = agent.invoke(
        initial_state,
        config={"configurable": {"thread_id": thread_id}},
        context=context,
    )

    # Extract the last AI message
    for msg in reversed(result.get("messages", [])):
        if hasattr(msg, "content") and msg.content and msg.type == "ai":
            return msg.content
    return str(result)


# ──────────────────────────────────────────────────────────────
# 9. EXAMPLE USAGE
# ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    requirements = """
    User Login Feature:
    - Users must log in with a valid email and password.
    - After 3 consecutive failed attempts, the account is locked for 15 minutes.
    - Users can reset their password via a link sent to their registered email.
    - Passwords must be at least 8 characters with 1 uppercase letter and 1 number.
    - Sessions expire after 30 minutes of inactivity.
    - Users see a descriptive error message for invalid credentials.
    - Remember-me option keeps the user logged in for 7 days.
    """

    output = generate_bdd_tests(
        requirements=requirements,
        project_name="AuthService",
        user_id="qa_engineer_01",
        thread_id="session-login-001",
        output_dir="./feature_files",
    )

    print("\n── Final Agent Response ──────────────────────────────")
    print(output)


  🤖 BDD Agent started  |  Session: session-login-001

  📞 Model call #1/20 | Messages in state: 1
  ✅ Model responded  | Call count now: 1 | Tools requested: ['parse_requirements']

  🔧 Tool call  → parse_requirements
       requirements: User Login Feature:
    - Users must log in with a valid email and password.
   …
  ✔  parse_requirements completed successfully.
  📞 Model call #2/20 | Messages in state: 3


e:\Akhil\PodTest\Clients\Self\Bootcamp\FullStackSDET\FSLangChain\myenv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BDDContext(user_id='qa_en...es', max_model_calls=20), input_type=BDDContext])
  return self.__pydantic_serializer__.to_python(


  ✅ Model responded  | Call count now: 2 | Tools requested: ['generate_bdd_scenarios']

  🔧 Tool call  → generate_bdd_scenarios
       parsed_requirements: Users must log in with valid email and password. Account locking after 3 failed …
  ✔  generate_bdd_scenarios completed successfully.
  📞 Model call #3/20 | Messages in state: 5


e:\Akhil\PodTest\Clients\Self\Bootcamp\FullStackSDET\FSLangChain\myenv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BDDContext(user_id='qa_en...es', max_model_calls=20), input_type=BDDContext])
  return self.__pydantic_serializer__.to_python(


  ✅ Model responded  | Call count now: 3 | Tools requested: ['save_gherkin_to_state']

  🔧 Tool call  → save_gherkin_to_state
       gherkin_text: Feature: User Login and Authentication

  As a registered user
  I want to secur…
  ✔  save_gherkin_to_state completed successfully.
  📞 Model call #4/20 | Messages in state: 7


e:\Akhil\PodTest\Clients\Self\Bootcamp\FullStackSDET\FSLangChain\myenv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BDDContext(user_id='qa_en...es', max_model_calls=20), input_type=BDDContext])
  return self.__pydantic_serializer__.to_python(


  ✅ Model responded  | Call count now: 4 | Tools requested: ['validate_gherkin_syntax']

  🔧 Tool call  → validate_gherkin_syntax
       gherkin_text: Feature: User Login and Authentication

  As a registered user
  I want to secur…
  ✔  validate_gherkin_syntax completed successfully.
  📞 Model call #5/20 | Messages in state: 9


e:\Akhil\PodTest\Clients\Self\Bootcamp\FullStackSDET\FSLangChain\myenv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=BDDContext(user_id='qa_en...es', max_model_calls=20), input_type=BDDContext])
  return self.__pydantic_serializer__.to_python(


  ✅ Model responded  | Call count now: 5 | Tools requested: ['export_feature_file']

  🔧 Tool call  → export_feature_file
       gherkin_text: Feature: User Login and Authentication

  As a registered user
  I want to secur…
  ✔  export_feature_file completed successfully.
  📞 Model call #6/20 | Messages in state: 11
  ✅ Model responded  | Call count now: 6

  📋 BDD Agent Session Summary
  Total model calls : 6
  Feature file      : ./feature_files\authservice_bdd_tests.feature
  Validation        : ✅ VALID — All Gherkin syntax checks passed.


── Final Agent Response ──────────────────────────────
✅ **BDD Test Cases Generated Successfully!**

**Feature File Exported To:**  
`./feature_files\authservice_bdd_tests.feature`

**Summary:**
- **17 Scenarios** covering:
  - ✅ **Happy paths** (successful login, valid password reset, session persistence)
  - ❌ **Negative/error scenarios** (invalid login, locked account, invalid email/password)
  - 🔒 **Security** (lockout after 3 failures, sess